# Lab: Sentiment Analysis  
#  *******Data-Centric vs Model-Centric approaches




This lab gives an introduction to sentiment analysis approaches.

In this lab, we'll build a classifier for product reviews (restricted to the magazine category), like:

> Excellent! I look forward to every issue. I had no idea just how much I didn't know.  The letters from the subscribers are educational, too.

Label: ⭐️⭐️⭐️⭐️⭐️ (good)

> My son waited and waited, it took the 6 weeks to get delivered that they said it would but when it got here he was so dissapointed, it only took him a few minutes to read it.

Label: ⭐️ (bad)

We'll work with a dataset that has some issues, and we'll see how we can squeeze only so much performance out of the model by being clever about model choice, searching for better hyperparameters, etc. Then, we'll take a look at the data (as any good data scientist should), develop an understanding of the issues, and use simple approaches to improve the data. Finally, we'll see how improving the data can improve results.

## Installing software

For this lab, you'll need to install [scikit-learn](https://scikit-learn.org/) and [pandas](https://pandas.pydata.org/). If you don't have them installed already, you can install them by running the following cell:

In [ ]:
!pip install scikit-learn pandas

# Loading the data

First, let's load the train/test sets and take a look at the data.

In [2]:
import pandas as pd

In [4]:
train = pd.read_csv('reviews_train.csv')
test = pd.read_csv('reviews_test.csv')

test.sample(5)

,review,label
669,I am unable to access the magazine. I'm being ...,bad
372,love this magazine !!!,good
125,"One of my favorite magazines. Great cooking, d...",good
453,"Consistently good articles, notes, fiction, an...",good
222,This is perhaps my favorite magazine of this t...,good


# Training a baseline model

There are many approaches for training a sequence classification model for text data. In this lab, we're giving you code that mirrors what you find if you look up [how to train a text classifier](https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html), where we'll train an SVM on [tf-idf](https://en.wikipedia.org/wiki/Tf%E2%80%93idf) features (numeric representations of each text field based on word occurrences).

In [5]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline

In [6]:
sgd_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', SGDClassifier()),
])

In [7]:
_ = sgd_clf.fit(train['review'], train['label'])

## Evaluating model accuracy

In [8]:
from sklearn import metrics

In [9]:
def evaluate(clf):
    pred = clf.predict(test['review'])
    acc = metrics.accuracy_score(test['label'], pred)
    print(f'Accuracy: {100*acc:.1f}%')

In [10]:
evaluate(sgd_clf)

Accuracy: 76.1%


## Trying another model

76% accuracy is not great for this binary classification problem. Can you do better with a different model, or by tuning hyperparameters for the SVM trained with SGD?

# Exercise 1

Can you train a more accurate model on the dataset (without changing the dataset)? You might find this [scikit-learn classifier comparison](https://scikit-learn.org/stable/auto_examples/classification/plot_classifier_comparison.html) handy, as well as the [documentation for supervised learning in scikit-learn](https://scikit-learn.org/stable/supervised_learning.html).

One idea for a model you could try is a [naive Bayes classifier](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html).

You could also try experimenting with different values of the model hyperparameters, perhaps tuning them via a [grid search](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html).

Or you can even try training multiple different models and [ensembling their predictions](https://scikit-learn.org/stable/modules/ensemble.html#voting-classifier), a strategy often used to win prediction competitions like Kaggle.

**Advanced:** If you want to be more ambitious, you could try an even fancier model, like training a Transformer neural network. If you go with that, you'll want to fine-tune a pre-trained model. This [guide from HuggingFace](https://huggingface.co/docs/transformers/training) may be helpful.

In [1]:
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def compare_models(models, X_train, y_train, X_test, y_test):
    """
    Compare multiple ML models on the same train/test split.

    Parameters
    ----------
    models : dict or list
        Either:
        - dict: {"Model Name": model_object, ...}
        - list: [model_object1, model_object2, ...]
    X_train, y_train : training data
    X_test, y_test : testing data

    Returns
    -------
    pd.DataFrame
        Table with model name and evaluation metrics.
    """

    # If user passes a list, convert it to a dict automatically
    if isinstance(models, list):
        models = {model.__class__.__name__: model for model in models}

    results = []

    for model_name, model in models.items():
        clf = clone(model)   # keeps original model untouched
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        results.append({
            "Model": model_name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, pos_label="good", zero_division=0),
            "Recall": recall_score(y_test, y_pred, pos_label="good", zero_division=0),
            "F1-score": f1_score(y_test, y_pred, pos_label="good", zero_division=0)
        })

    results_df = pd.DataFrame(results).sort_values(by="F1-score", ascending=False).reset_index(drop=True)
    return results_df

In [11]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.svm import LinearSVC

models = {
    "Naive Bayes": Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("clf", MultinomialNB())
    ]),

    "SGD Classifier": Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("clf", SGDClassifier(random_state=42))
    ]),

    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("clf", LogisticRegression(max_iter=1000))
    ]),

    "Linear SVC": Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("clf", LinearSVC())
    ])
}

results_df = compare_models(
    models,
    train["review"], train["label"],
    test["review"], test["label"]
)

results_df

,Model,Accuracy,Precision,Recall,F1-score
0,Naive Bayes,0.853,0.853707,0.852,0.852853
1,Logistic Regression,0.781,0.767619,0.806,0.786341
2,SGD Classifier,0.763,0.761431,0.766,0.763709
3,Linear SVC,0.720,0.711538,0.740,0.725490


## Taking a closer look at the training data

Let's actually take a look at some of the training data:

In [12]:
train.head()

,review,label
0,Based on all the negative comments about Taste...,good
1,I still have not received this. Obviously I c...,bad
2,</tr>The magazine is not worth the cost of sub...,good
3,This magazine is basically ads. Kindve worthle...,bad
4,"The only thing I've recieved, so far, is the b...",bad


Zooming in on one particular data point:

In [14]:
print(train.iloc[0].to_dict())

{'review': "Based on all the negative comments about Taste of Home, I will not subscribeto the magazine. In the past it was a great read.\nSorry it, too, has gone the 'way of the wind'.<br>o-p28pass4 </br>", 'label': 'good'}


This data point is labeled "good", but it's clearly a negative review. Also, it looks like there's some funny HTML stuff at the end.

# Exercise 2

Take a look at some more examples in the dataset. Do you notice any patterns with bad data points?

In [15]:
train[train['review'].str.contains(r'<[^>]+>', regex=True, na=False)].sample(10)

,review,label
4789,</div>Just not a fan of this magazine,good
2868,<li>contentEdgeGreat magazine. Had been a subs...,bad
268,Unfortunately I rarely receive a copy.</HEAD>,good
2808,"<head>I love this magazine, so many great idea...",bad
3104,I never received the free clutch. The magazine...,good
5616,"Cover article was a person's opinion, not the ...",good
4715,</dd>I really enjoy this magazine. Lots of ne...,bad
6328,"</html>A must-read, each & every week!",bad
2322,<DL>Nothing like I thought it would be like</li>,good
5494,The amount of interesting articles declined. I...,good


## Issues in the data

It looks like there's some funny HTML tags in our dataset, and those datapoints have nonsense labels. Maybe this dataset was collected by scraping the internet, and the HTML wasn't quite parsed correctly in all cases.

# Exercise 3

To address this, a simple approach we might try is to throw out the bad data points, and train our model on only the "clean" data.

Come up with a simple heuristic to identify data points containing HTML, and filter out the bad data points to create a cleaned training set.

In [25]:
import re

def is_bad_data(review):
    return bool(re.search(r"<[^>]+>", str(review)))

## Creating the cleaned training set

In [26]:
train_clean = train[~train["review"].apply(is_bad_data)].copy()

print("Original training size:", len(train))
print("Cleaned training size :", len(train_clean))
print("Removed rows          :", len(train) - len(train_clean))

Original training size: 6666
Cleaned training size : 4018
Removed rows          : 2648


In [27]:
sgd_clf.fit(train_clean["review"], train_clean["label"])
pred_clean = sgd_clf.predict(test["review"])

In [29]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy:", accuracy_score(test["label"], pred_clean))
print("Precision:", precision_score(test["label"], pred_clean, pos_label="good", zero_division=0))
print("Recall:", recall_score(test["label"], pred_clean, pos_label="good", zero_division=0))
print("F1-score:", f1_score(test["label"], pred_clean, pos_label="good", zero_division=0))

Accuracy: 0.968
Precision: 0.9717741935483871
Recall: 0.964
F1-score: 0.9678714859437751


## Evaluating a model trained on the clean training set

In [20]:
from sklearn import clone

In [30]:
sgd_clf_clean = clone(sgd_clf)

In [31]:
_ = sgd_clf_clean.fit(train_clean['review'], train_clean['label'])

This model should do significantly better:

In [32]:
evaluate(sgd_clf_clean)

Accuracy: 97.0%
